# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shwetabh1013/flyrank-ml-internship-shwetabh/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Two signal checks first -- do the underlying signals actually behave the way the flag logic assumes?**

**Signal A -- staleness (behind the refresh flags).** Claim: "the longer since a page was last updated, the more likely it is declining." Test: bucket by `freshness_tier`, look at the share of rows with `trend_direction == 'down'` in each bucket (n printed below).

**Signal B -- CTR vs. position (behind the CTR-fix logic).** Claim: "the better a page's average search position, the higher its click-through rate." Test: bucket by `position_tier`, compute the *weighted* CTR (total clicks / total impressions, not the mean of per-page CTRs -- the auditing-signals skill's trap) in each bucket (n printed below).

**My rule, in plain words:** A page belongs at the top of the refresh queue if it's ranking well enough that clicks should be flowing (`top_3`/`page_1`/`striking`/`page_3_5`) but its own CTR is less than half of what pages in that same position band typically get -- that gap is a metadata/snippet problem a refresh can fix -- **and** the page hasn't been touched in 91+ days, so the content itself is also due a look, not just the title tag. Both conditions have to hold; this is one rule, not three stacked ones.

**Reason code (one, constant across every flagged row):** `stale_and_ctr_underperform` -- a row either matches this pattern or it doesn't; there's no second reason code for this baseline.

**Action label:** `refresh_priority` for anything the rule flags (score > 0); everything else gets `no_action` and sits at the bottom of the ranked file.

In [1]:
import pandas as pd
from pathlib import Path

candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
csv_path = next(p for p in candidates if p.exists())
df = pd.read_csv(csv_path)

# --- Signal A: staleness vs. declining rate ---
freshness_order = ["0-30", "31-90", "91-180", "181+"]
signal_a = (
    df.groupby("freshness_tier")
      .agg(n=("content_id", "size"),
           declining_rate_pct=("trend_direction", lambda s: round((s == "down").mean() * 100, 1)))
      .reindex(freshness_order)
)
print("Signal A -- freshness_tier vs. % declining (trend_direction == \'down\')")
print(signal_a)
print()

# --- Signal B: position vs. weighted CTR ---
position_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
pos_df = df[df["position_tier"].isin(position_order)]
signal_b = pos_df.groupby("position_tier").apply(
    lambda g: pd.Series({
        "n": len(g),
        "weighted_ctr_pct": round(g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100, 3),
    })
).reindex(position_order)
print("Signal B -- position_tier vs. weighted CTR (total clicks / total impressions)")
print(signal_b)


Signal A -- freshness_tier vs. % declining (trend_direction == 'down')
                    n  declining_rate_pct
freshness_tier                           
0-30            20480                51.1
31-90             175                58.9
91-180           9171                61.1
181+              174                47.1

Signal B -- position_tier vs. weighted CTR (total clicks / total impressions)
                     n  weighted_ctr_pct
position_tier                           
top_3           2321.0             0.488
page_1         11814.0             0.350
striking        7304.0             0.347
page_3_5        7242.0             0.155
deep            1319.0             0.041


**Verdicts**

- **Signal A -- MIXED.** Declining rate rises with staleness through the middle tiers (0-30: 51.1%, n=20,480 -> 31-90: 58.9%, n=175 -> 91-180: 61.1%, n=9,171) but then *drops back* to 47.1% (n=174) for `181+` -- the most-stale bucket looks the healthiest by this measure, not the worst. All four buckets clear the ~50-row floor, so this isn't a tiny-cell fluke; it's a real reversal. My read: pages that have been stale for 181+ days and still show `down` would mostly have bottomed out already and settled into `stable` -- a floor effect, not evidence that extreme staleness is safe. So I'm gating my rule on a moderate staleness threshold (91+ days) rather than trusting a clean straight line across all four tiers.

- **Signal B -- CONFIRMED.** Weighted CTR falls in lockstep with position quality: `top_3` 0.488% (n=2,321) -> `page_1` 0.350% (n=11,814) -> `striking` 0.347% (n=7,304) -> `page_3_5` 0.155% (n=7,242) -> `deep` 0.041% (n=1,319). Clean and monotonic -- this is the signal my rule's CTR-gap half leans on.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# tier CTR benchmark, computed the same weighted way as Signal B above
position_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
tier_ctr = (
    df[df["position_tier"].isin(position_order)]
      .groupby("position_tier")
      .apply(lambda g: g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100)
)
df["tier_ctr_benchmark"] = df["position_tier"].map(tier_ctr)

# the rule -- readable on purpose, no fitted weights
stale = (df["days_since_last_update"] >= 91).astype(int)
ctr_gap = (
    (df["ctr"] < 0.5 * df["tier_ctr_benchmark"]) & df["position_tier"].isin(position_order)
).astype(int)

df["score"] = stale * ctr_gap * df["impressions_90d"]
df["reason_code"] = "stale_and_ctr_underperform"
df["action"] = df["score"].apply(lambda s: "refresh_priority" if s > 0 else "no_action")

n_flagged = (df["score"] > 0).sum()
print(f"rows flagged for the queue: {n_flagged:,} / {len(df):,}")

queue_cols = [
    "content_id", "client_id", "content_type", "position_tier", "avg_position",
    "ctr", "tier_ctr_benchmark", "days_since_last_update", "freshness_tier",
    "impressions_90d", "clicks_90d", "trend_direction", "score", "reason_code", "action",
]
queue = df[queue_cols].sort_values("score", ascending=False).reset_index(drop=True)

# resolve via the real (symlink-free) location of the data file, so this lands
# in work/outputs/ correctly whether we run from repo root or from work/notebooks/
repo_root = csv_path.resolve().parents[2]
out_path = repo_root / "work/outputs/baseline_action_score.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(out_path, index=False)
print(f"wrote {out_path} -- {len(queue):,} rows")
queue.head(10)


rows flagged for the queue: 5,245 / 30,000


wrote /home/claude/work/repo/work/outputs/baseline_action_score.csv -- 30,000 rows


,content_id,client_id,content_type,position_tier,avg_position,ctr,tier_ctr_benchmark,days_since_last_update,freshness_tier,impressions_90d,clicks_90d,trend_direction,score,reason_code,action
0,content_5fe46e04994d,client_4e07408562,keyword article,page_1,4.2,0.14,0.350324,104,91-180,517715,741,down,517715,stale_and_ctr_underperform,refresh_priority
1,content_cb112fce36be,client_19581e27de,keyword article,page_1,5.6,0.16,0.350324,104,91-180,309910,492,down,309910,stale_and_ctr_underperform,refresh_priority
2,content_36ff89c8214e,client_19581e27de,keyword article,page_1,7.3,0.05,0.350324,104,91-180,295097,154,stable,295097,stale_and_ctr_underperform,refresh_priority
3,content_b28d1efd668f,client_6208ef0f77,keyword article,page_3_5,26.2,0.06,0.154905,104,91-180,286608,169,stable,286608,stale_and_ctr_underperform,refresh_priority
4,content_813e88069237,client_6208ef0f77,keyword article,page_3_5,26.2,0.06,0.154905,104,91-180,233561,129,down,233561,stale_and_ctr_underperform,refresh_priority
5,content_c8e9d6ab9013,client_19581e27de,keyword article,page_1,9.7,0.00,0.350324,104,91-180,208678,0,down,208678,stale_and_ctr_underperform,refresh_priority
6,content_a7427266c305,client_19581e27de,keyword article,page_1,5.7,0.11,0.350324,104,91-180,201111,219,stable,201111,stale_and_ctr_underperform,refresh_priority
7,content_33b4dceecad1,client_19581e27de,keyword article,page_1,6.2,0.16,0.350324,104,91-180,181574,295,stable,181574,stale_and_ctr_underperform,refresh_priority
8,content_91652435f57a,client_19581e27de,keyword article,page_1,7.8,0.06,0.350324,104,91-180,159590,100,stable,159590,stale_and_ctr_underperform,refresh_priority
9,content_f42eb861c6dd,client_19581e27de,keyword article,page_1,6.5,0.13,0.350324,104,91-180,152467,191,down,152467,stale_and_ctr_underperform,refresh_priority


## 3. Top-10 review

*For each of your top ten, one line each -- the action, why it's there, and what would make it wrong.*

In [3]:
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
top10 = queue.head(10).copy()
top10


,content_id,client_id,content_type,position_tier,avg_position,ctr,tier_ctr_benchmark,days_since_last_update,freshness_tier,impressions_90d,clicks_90d,trend_direction,score,reason_code,action
0,content_5fe46e04994d,client_4e07408562,keyword article,page_1,4.2,0.14,0.350324,104,91-180,517715,741,down,517715,stale_and_ctr_underperform,refresh_priority
1,content_cb112fce36be,client_19581e27de,keyword article,page_1,5.6,0.16,0.350324,104,91-180,309910,492,down,309910,stale_and_ctr_underperform,refresh_priority
2,content_36ff89c8214e,client_19581e27de,keyword article,page_1,7.3,0.05,0.350324,104,91-180,295097,154,stable,295097,stale_and_ctr_underperform,refresh_priority
3,content_b28d1efd668f,client_6208ef0f77,keyword article,page_3_5,26.2,0.06,0.154905,104,91-180,286608,169,stable,286608,stale_and_ctr_underperform,refresh_priority
4,content_813e88069237,client_6208ef0f77,keyword article,page_3_5,26.2,0.06,0.154905,104,91-180,233561,129,down,233561,stale_and_ctr_underperform,refresh_priority
5,content_c8e9d6ab9013,client_19581e27de,keyword article,page_1,9.7,0.00,0.350324,104,91-180,208678,0,down,208678,stale_and_ctr_underperform,refresh_priority
6,content_a7427266c305,client_19581e27de,keyword article,page_1,5.7,0.11,0.350324,104,91-180,201111,219,stable,201111,stale_and_ctr_underperform,refresh_priority
7,content_33b4dceecad1,client_19581e27de,keyword article,page_1,6.2,0.16,0.350324,104,91-180,181574,295,stable,181574,stale_and_ctr_underperform,refresh_priority
8,content_91652435f57a,client_19581e27de,keyword article,page_1,7.8,0.06,0.350324,104,91-180,159590,100,stable,159590,stale_and_ctr_underperform,refresh_priority
9,content_f42eb861c6dd,client_19581e27de,keyword article,page_1,6.5,0.13,0.350324,104,91-180,152467,191,down,152467,stale_and_ctr_underperform,refresh_priority


1. `content_5fe46e04994d` -- **refresh_priority**. `page_1` (avg pos 4.2) but CTR 0.14% vs. a 0.350% tier benchmark, 517,715 impressions, 104 days since update, `trend_direction = down`. Cleanest case in the queue: good position, real traffic, clear CTR gap, and it's actually declining. Would be wrong if the low CTR is a SERP-feature issue (a snippet or ad above it stealing clicks) rather than a fixable title/meta -- that needs a manual SERP check.
2. `content_cb112fce36be` -- **refresh_priority**. Same client and 104-day staleness, `page_1` CTR 0.16% vs. 0.350%, 309,910 impressions, `down`. Same logic as #1. Would be wrong if this page and #1 share a template bug (both mis-render meta descriptions) -- one shared fix could resolve both, so counting them as two separate priorities double-counts the effort.
3. `content_36ff89c8214e` -- **refresh_priority**. `page_1` (avg pos 7.3), CTR 0.05% vs. 0.350% -- the biggest gap in the top 10 -- 295,097 impressions, but `trend_direction = stable`, not `down`. The CTR gap is real; it just isn't losing traffic, it never captured it. Would be wrong to pitch this as "recover lost traffic" -- it's "capture traffic that was never captured," a different story for the editor.
4. `content_b28d1efd668f` -- **refresh_priority**. `page_3_5` (avg pos 26.2), CTR 0.06% vs. 0.155% benchmark, 286,608 impressions, `stable`. Weaker position than #1-3, so the gap matters less in absolute terms even though the ratio check still passes. Would be wrong if the real lever here is a ranking fix, not a CTR/snippet fix -- CTR optimization has limited upside until the page climbs out of `page_3_5`.
5. `content_813e88069237` -- **refresh_priority**. Same position tier and CTR-gap pattern as #4, 233,561 impressions, `down` this time -- same read as #4, one more piece of evidence behind it.
6. `content_c8e9d6ab9013` -- **refresh_priority**. `page_1` (avg pos 9.7), CTR **0.00%**, `clicks_90d = 0` against 208,678 impressions, `down`. Zero clicks on that much volume is unusual even for a bad snippet. Would be wrong to treat this as an ordinary CTR-fix without first checking for a technical problem (broken canonical, accidental noindex, redirect loop) -- a metadata refresh won't fix a page Google isn't actually surfacing to users.
7. `content_a7427266c305` -- **refresh_priority**. `page_1` (avg pos 5.7), CTR 0.11% vs. 0.350%, 201,111 impressions, `stable`. Same "gap is real, decline isn't" caveat as #3.
8. `content_33b4dceecad1` -- **refresh_priority**. `page_1` (avg pos 6.2), CTR 0.16% vs. 0.350%, 181,574 impressions, `stable`. Same caveat as #3/#7.
9. `content_91652435f57a` -- **refresh_priority**. `page_1` (avg pos 7.8), CTR 0.06% vs. 0.350%, 159,590 impressions, `stable`. Same caveat again -- three of ten are `stable`, not `down`.
10. `content_f42eb861c6dd` -- **refresh_priority**. `page_1` (avg pos 6.5), CTR 0.13% vs. 0.350%, 152,467 impressions, `down`. Same pattern as #1/#2/#5.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks, named**

- **3 of the top 10 (#3, #7, #9) are `trend_direction == 'stable'`, not `down`.** My rule doesn't check trend at all -- it ranks purely on CTR-gap x staleness x impressions. That's honest to what the rule actually measures (a metadata opportunity, not a decline), but if I pitched this queue as "pages losing traffic," those three would be the wrong evidence for that pitch. Fix isn't to add `trend_direction` as a weight (that's the label, off-limits) -- it's to describe the action precisely instead of overselling it.

- **`content_c8e9d6ab9013` (#6) has 0 clicks on 208,678 impressions.** Not a normal CTR gap -- a near-total absence of clicks, a different failure mode (possible indexing/canonicalization issue) that a content refresh alone won't fix. A real workflow needs a check that routes obvious-zero rows to "investigate" instead of straight to "refresh."

- **The staleness signal is close to a constant inside the flagged group.** 8,773 of the 9,171 rows in the `91-180` freshness tier share the exact same `days_since_last_update` value (104) -- almost certainly one bulk-update event in this teaching slice, not thousands of pages independently drifting stale. That means the ranking inside my flagged set is really being driven by impressions and the CTR gap, not by any staleness gradation -- I should say that plainly rather than imply staleness is doing more differentiating work than it is.

- **No per-client normalization.** 11 of the top 20 rows belong to a single client (`client_19581e27de`). The score is raw `impressions_90d`, so a large, high-traffic client dominates the top of the queue regardless of how urgent any one page is *relative to that client's own baseline*. A client-normalized score (percentile within `client_id`) would be a fairer next version, and is exactly the kind of adjustment a trained model could learn instead of me guessing a normalization scheme by hand.

**Leakage check**

Confirms this rule uses only observed, current-window fields -- nothing from a future outcome, and none of FlyRank's own product flags (per the flyrank-context skill: `health_score`, `needs_ctr_fix`, `is_quick_win`, `priority_score`, `action_type` are outputs, never inputs).

In [4]:
rule_inputs = {"days_since_last_update", "ctr", "position_tier", "impressions_90d"}
banned_as_inputs = {
    "trend_direction", "trend_pct",                       # label source
    "is_declining_label",                                  # the target itself
    "health_score", "needs_ctr_fix", "is_quick_win",       # FlyRank product flags -- outputs, never inputs
    "priority_score", "action_type", "refresh_tier",
}

overlap = rule_inputs & banned_as_inputs
print("rule inputs:", sorted(rule_inputs))
print("banned columns used by the rule:", overlap if overlap else "none")
print()
print("all rule inputs are current-window observed fields from the same 90-day snapshot")
print("the whole CSV is built from -- no future window, no product flag, no label column.")


rule inputs: ['ctr', 'days_since_last_update', 'impressions_90d', 'position_tier']
banned columns used by the rule: none

all rule inputs are current-window observed fields from the same 90-day snapshot
the whole CSV is built from -- no future window, no product flag, no label column.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.